<a href="https://colab.research.google.com/github/joshiapoorv/AI-Legal-Document-Assistant/blob/main/AI_LEGAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Dependencies**

In [ ]:
!pip install -q gradio pymongo pypdf bcrypt numpy python-dotenv transformers torch accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.4 MB/s eta 0:00:00


In [ ]:
import os

PROJECT_DIR = "/content/legal-rag-python"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder created:", PROJECT_DIR)

Project folder created: /content/legal-rag-python


In [ ]:
from urllib.parse import quote_plus
from pathlib import Path

PROJECT_DIR = "/content/legal-rag-python"

DB_USER = "vaibhavnaithani010906_db_user"
DB_PASS = "PBp58W47gqZdmeeH"
CLUSTER_HOST = "cluster0.xj3gg8x.mongodb.net"
DB_NAME = "legal"

encoded_user = quote_plus(DB_USER)
encoded_pass = quote_plus(DB_PASS)

MONGO_URI = (
    f"mongodb+srv://{encoded_user}:{encoded_pass}@{CLUSTER_HOST}/{DB_NAME}"
    "?retryWrites=true&w=majority&tls=true"
)

env_text = f"""MONGO_URI={MONGO_URI}
DB_NAME={DB_NAME}
LEGAL_BERT_MODEL=nlpaueb/legal-bert-base-uncased
GENERATION_MODEL=google/flan-t5-base
"""

Path(f"{PROJECT_DIR}/.env").write_text(env_text)

print(".env created successfully")

.env created successfully


In [ ]:
import os
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv("/content/legal-rag-python/.env")

MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
client.admin.command("ping")

print("MongoDB connected successfully")

MongoDB connected successfully


In [ ]:
%%writefile /content/legal-rag-python/app.py
import os
import re
import uuid
import bcrypt
import numpy as np
import gradio as gr
import torch

from datetime import datetime
from pypdf import PdfReader
from pymongo import MongoClient
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM

load_dotenv("/content/legal-rag-python/.env")

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("DB_NAME", "legal_rag_assistant")

LEGAL_BERT_MODEL = os.getenv("LEGAL_BERT_MODEL", "nlpaueb/legal-bert-base-uncased")
GENERATION_MODEL = os.getenv("GENERATION_MODEL", "google/flan-t5-base")

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

users_col = db["users"]
documents_col = db["documents"]
chunks_col = db["document_chunks"]
chats_col = db["chat_history"]

print("Loading LegalBERT embedding model...")
legal_tokenizer = AutoTokenizer.from_pretrained(LEGAL_BERT_MODEL)
legal_model = AutoModel.from_pretrained(LEGAL_BERT_MODEL)
legal_model.eval()

print("Loading FLAN-T5 answer generation model...")
gen_tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)
gen_model.eval()

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def extract_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    return clean_text(text)

def chunk_text(text, chunk_size=900, overlap=160):
    text = clean_text(text)
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

def get_legalbert_embedding(text):
    inputs = legal_tokenizer(
        text[:3500],
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        outputs = legal_model(**inputs)

    attention_mask = inputs["attention_mask"]
    token_embeddings = outputs.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    masked_embeddings = token_embeddings * mask

    summed = torch.sum(masked_embeddings, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)

    embedding = (summed / counts).squeeze().numpy()

    return embedding.tolist()

def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)

    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator == 0:
        return 0.0

    return float(np.dot(a, b) / denominator)

def generate_answer(prompt):
    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def seed_demo_users():
    users_col.delete_many({})
    documents_col.delete_many({})
    chunks_col.delete_many({})
    chats_col.delete_many({})

    demo_users = [
        ["Demo Lawyer", "lawyer@example.com", "lawyer123", "lawyer"],
        ["Demo Law Student", "student@example.com", "student123", "student"],
        ["Demo Client", "client@example.com", "client123", "client"]
    ]

    for name, email, password, role in demo_users:
        password_hash = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

        users_col.insert_one({
            "name": name,
            "email": email,
            "password_hash": password_hash,
            "role": role,
            "created_at": datetime.utcnow()
        })

    return "Demo users created. Login with lawyer@example.com / lawyer123"

def login(email, password):
    user = users_col.find_one({"email": email.strip().lower()})

    if not user:
        return None, "Invalid email or password"

    valid = bcrypt.checkpw(password.encode(), user["password_hash"].encode())

    if not valid:
        return None, "Invalid email or password"

    session = {
        "name": user["name"],
        "email": user["email"],
        "role": user["role"]
    }

    return session, f"Logged in as {user['name']} ({user['role']})"

def upload_pdf(session, pdf_file, title, document_type):
    try:
        if not session:
            return "Please login first."

        if pdf_file is None:
            return "Please upload a PDF file."

        text = extract_pdf_text(pdf_file.name)

        if len(text) < 50:
            return "Could not extract enough text. Use a text-based PDF, not a scanned image PDF."

        chunks = chunk_text(text)
        document_id = str(uuid.uuid4())
        filename = os.path.basename(pdf_file.name)

        documents_col.insert_one({
            "document_id": document_id,
            "title": title or filename,
            "document_type": document_type,
            "filename": filename,
            "uploaded_by": session["email"],
            "total_chunks": len(chunks),
            "text_preview": text[:500],
            "created_at": datetime.utcnow()
        })

        for index, chunk in enumerate(chunks):
            embedding = get_legalbert_embedding(chunk)

            chunks_col.insert_one({
                "document_id": document_id,
                "filename": filename,
                "chunk_index": index,
                "text": chunk,
                "embedding": embedding,
                "uploaded_by": session["email"],
                "created_at": datetime.utcnow()
            })

        return f"Upload successful. Created {len(chunks)} chunks and LegalBERT embeddings."

    except Exception as e:
        return f"Upload error: {str(e)}"

def list_documents():
    docs = list(documents_col.find({}, {"_id": 0}).sort("created_at", -1))

    if not docs:
        return "No documents uploaded yet.", gr.update(choices=["All Documents"], value="All Documents")

    choices = ["All Documents"]
    lines = []

    for doc in docs:
        choices.append(f"{doc['document_id']}::{doc['title']}")
        lines.append(
            f"Title: {doc['title']}\nType: {doc['document_type']}\nChunks: {doc['total_chunks']}\nFile: {doc['filename']}\n"
        )

    return "\n".join(lines), gr.update(choices=choices, value=choices[0])

def retrieve_chunks(question, selected_document):
    question_embedding = get_legalbert_embedding(question)

    query = {}

    if selected_document and selected_document != "All Documents":
        document_id = selected_document.split("::")[0]
        query = {"document_id": document_id}

    chunks = list(chunks_col.find(query, {"_id": 0}))
    scored = []

    for chunk in chunks:
        chunk["score"] = cosine_similarity(question_embedding, chunk["embedding"])
        scored.append(chunk)

    scored.sort(key=lambda item: item["score"], reverse=True)

    return scored[:5]

def ask_question(session, question, selected_document):
    try:
        if not session:
            return "Please login first."

        if not question.strip():
            return "Please enter a question."

        top_chunks = retrieve_chunks(question, selected_document)

        if not top_chunks:
            return "No document chunks found. Upload a legal PDF first."

        context = "\n\n".join([
            f"Source {i + 1}: {chunk['text']}"
            for i, chunk in enumerate(top_chunks)
        ])

        prompt = f"""
Answer the legal question using only the context.

Context:
{context}

Question:
{question}

Instructions:
Do not invent facts.
Use simple language.
End with this sentence: This is not professional legal advice.

Answer:
"""

        answer = generate_answer(prompt)

        sources = [
            f"{chunk['filename']} - chunk {chunk['chunk_index']} - score {chunk['score']:.4f}"
            for chunk in top_chunks
        ]

        chats_col.insert_one({
            "user_email": session["email"],
            "question": question,
            "answer": answer,
            "sources": sources,
            "created_at": datetime.utcnow()
        })

        return answer + "\n\nSources:\n" + "\n".join(sources)

    except Exception as e:
        return f"Answer error: {str(e)}"

def summarize_document(session, selected_document):
    if not session:
        return "Please login first."

    if not selected_document or selected_document == "All Documents":
        return "Please select a specific document."

    return ask_question(
        session,
        "Summarize the document. Include parties, obligations, rights, risks, dates, penalties, and termination terms.",
        selected_document
    )

def view_history(session):
    if not session:
        return "Please login first."

    history = list(
        chats_col.find({"user_email": session["email"]}, {"_id": 0})
        .sort("created_at", -1)
        .limit(10)
    )

    if not history:
        return "No chat history yet."

    return "\n\n---\n\n".join([
        f"Question: {item['question']}\n\nAnswer: {item['answer']}\n\nSources: {item['sources']}"
        for item in history
    ])

def delete_documents(session):
    if not session:
        return "Please login first."

    documents_col.delete_many({})
    chunks_col.delete_many({})

    return "All documents and embeddings deleted."

with gr.Blocks(title="AI Powered Legal Document Assistant") as demo:
    session_state = gr.State(None)

    gr.Markdown("# AI Powered Legal Document Assistant")
    gr.Markdown(
        """
        This project uses **LegalBERT transformer embeddings** with a **RAG pipeline** for legal PDF question answering.

        Features:
        - Login system
        - PDF upload
        - Legal text extraction
        - Chunking
        - LegalBERT embeddings
        - MongoDB storage
        - Semantic retrieval
        - AI answer generation
        - Document summary
        - Chat history
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("## Login")

            email = gr.Textbox(label="Email", value="lawyer@example.com")
            password = gr.Textbox(label="Password", value="lawyer123", type="password")

            login_btn = gr.Button("Login")
            seed_btn = gr.Button("Reset / Seed Demo Users")

            login_status = gr.Textbox(label="Login Status")

        with gr.Column():
            gr.Markdown("## Upload Legal PDF")

            pdf_file = gr.File(label="Upload PDF", file_types=[".pdf"])
            title = gr.Textbox(label="Document Title")

            document_type = gr.Dropdown(
                label="Document Type",
                choices=["contract", "agreement", "notice", "policy", "case_law", "petition", "other"],
                value="contract"
            )

            upload_btn = gr.Button("Upload and Embed PDF")
            upload_output = gr.Textbox(label="Upload Result", lines=5)

    gr.Markdown("## Documents")

    with gr.Row():
        refresh_btn = gr.Button("Refresh Documents")
        delete_btn = gr.Button("Delete All Documents")

    document_list = gr.Textbox(label="Uploaded Documents", lines=8)

    document_dropdown = gr.Dropdown(
        label="Select Document",
        choices=["All Documents"],
        value="All Documents"
    )

    gr.Markdown("## Ask Questions")

    question = gr.Textbox(
        label="Legal Question",
        lines=4,
        value="Summarize the key clauses, obligations, risks, and important dates in this document."
    )

    with gr.Row():
        ask_btn = gr.Button("Ask Question")
        summary_btn = gr.Button("Summarize Selected Document")
        history_btn = gr.Button("View Chat History")

    answer_output = gr.Textbox(label="Answer", lines=16)

    login_btn.click(login, inputs=[email, password], outputs=[session_state, login_status])
    seed_btn.click(seed_demo_users, outputs=login_status)

    upload_btn.click(
        upload_pdf,
        inputs=[session_state, pdf_file, title, document_type],
        outputs=upload_output
    ).then(
        list_documents,
        outputs=[document_list, document_dropdown]
    )

    refresh_btn.click(list_documents, outputs=[document_list, document_dropdown])
    ask_btn.click(ask_question, inputs=[session_state, question, document_dropdown], outputs=answer_output)
    summary_btn.click(summarize_document, inputs=[session_state, document_dropdown], outputs=answer_output)
    history_btn.click(view_history, inputs=session_state, outputs=answer_output)

    delete_btn.click(
        delete_documents,
        inputs=session_state,
        outputs=upload_output
    ).then(
        list_documents,
        outputs=[document_list, document_dropdown]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Writing /content/legal-rag-python/app.py


In [ ]:
%cd /content/legal-rag-python
!python app.py

/content/legal-rag-python
Loading LegalBERT embedding model...
config.json: 100% 1.02k/1.02k [00:00<00:00, 1.39MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 82.1kB/s]
vocab.txt: 100% 222k/222k [00:00<00:00, 12.9MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 291kB/s]

pytorch_model.bin: downloading bytes:  73% 321M/440M [00:03<00:01, 99.8MB/s, 26.6MB/s  ]
pytorch_model.bin: downloading bytes:  88% 386M/440M [00:04<00:00, 63.2MB/s, 29.8MB/s  ]
pytorch_model.bin: downloading bytes:  93% 412M/440M [00:05<00:00, 57.6MB/s, 30.3MB/s  ]
pytorch_model.bin: reconstructing file:  76% 335M/440M [00:05<00:01, 76.4MB/s, 23.4MB/s  ]
pytorch_model.bin: downloading bytes: 100% 415M/415M [00:06<00:00, 68.7MB/s, 31.1MB/s  ]
pytorch_model.bin: reconstructing file: 100% 440M/440M [00:06<00:00, 72.8MB/s, 36.9MB/s  ]

model.safetensors: downloading bytes:  32% 141M/440M [00:02<00:02, 131MB/s, 11.0MB/s  ]

Loading weights: 100% 199/199 [00:00<00:00, 2765.44it/s]
model.safetensors: downl